# LLaMA 3.2 3B

In [ ]:
# Cell 1 — Réinstaller proprement
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
print("Installation terminée! ✅")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-mvq2o6pp/unsloth_c5e232e624a54c9fbbc6fd27cdff9939
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-mvq2o6pp/unsloth_c5e232e624a54c9fbbc6fd27cdff9939
  Resolved https://github.com/unslothai/unsloth.git to commit eeb49d54b8d801a1ce922fa15f778e2bad205db0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 159.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 31.0 MB/s eta 0:00:00

In [ ]:
# Cell 2 — Training complet
# ← Unsloth DOIT être importé en premier
import unsloth
from unsloth import FastLanguageModel
from unsloth.trainer import UnslothTrainer, UnslothTrainingArguments

import torch
import os
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from huggingface_hub import login

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

login() # ton token Read
from getpass import getpass
HF_WRITE_TOKEN = getpass("Hugging Face write token: ") # ton token Write

print(f"GPU: {torch.cuda.get_device_name(0)} ✅")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Charger dataset v3 ────────────────────────────────
print("\nChargement dataset v3...")
dataset_v3 = load_dataset(
    "abdeldjalil-smaili/algerian-medical-llm-v3",
    token=HF_TOKEN
)
print(f"Train: {len(dataset_v3['train'])} samples ✅")
print(f"Test:  {len(dataset_v3['test'])} samples ✅")

# ── Charger LLaMA 3.2 3B avec Unsloth ────────────────
print("\nChargement LLaMA 3.2 3B avec Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "meta-llama/Llama-3.2-3B-Instruct",
    max_seq_length= 2048,
    dtype         = None,       # ← auto detection
    load_in_4bit  = False,      # ← 102GB VRAM → pas besoin
    token         = HF_TOKEN,
)

# ── LoRA avec Unsloth ─────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                       = 32,
    lora_alpha              = 64,
    target_modules          = ["q_proj", "k_proj", "v_proj", "o_proj",
                               "gate_proj", "up_proj", "down_proj"],
    lora_dropout            = 0,    # ← 0 pour fast patching Unsloth
    bias                    = "none",
    use_gradient_checkpointing = "unsloth",
    random_state            = 42,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"LLaMA 3.2 3B chargé! ✅")

# ── SFTConfig ─────────────────────────────────────────
sft_config = SFTConfig(
    output_dir                = "./llama3b-algerian-medical",
    num_train_epochs          = 3,
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 4,
    learning_rate             = 2e-4,
    fp16                      = False,
    bf16                      = True,
    logging_steps             = 50,
    eval_strategy             = "steps",
    eval_steps                = 500,
    save_steps                = 500,
    warmup_steps              = 100,
    lr_scheduler_type         = "cosine",
    load_best_model_at_end    = True,
    report_to                 = "none",
    max_length                = 2048,
    dataset_text_field        = "text",
    push_to_hub               = True,
    hub_model_id              = ""
    ""
    ""
    ""
    "/llama3b-algerian-medical",
    hub_token                 = HF_WRITE_TOKEN,
    hub_strategy              = "checkpoint",
)

# ── Trainer ───────────────────────────────────────────
trainer = SFTTrainer(
    model             = model,
    train_dataset     = dataset_v3['train'],
    eval_dataset      = dataset_v3['test'],
    args              = sft_config,
    processing_class  = tokenizer,
)

print("="*50)
print(f"Fine-tuning LLaMA 3.2 3B — Blackwell 102GB")
print(f"Train: {len(dataset_v3['train'])} samples")
print(f"Test:  {len(dataset_v3['test'])} samples")
print(f"Epochs: 3 | Batch: 8 | LoRA r=32 | bf16")
print("="*50)

trainer.train()

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition ✅
VRAM: 102.0 GB

Chargement dataset v3...


README.md:   0%|          | 0.00/378 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/13.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/91138 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10127 [00:00<?, ? examples/s]

... (17 outputs left out)

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
500,1.864815,1.839366
1000,1.744601,1.730950
1500,1.693402,1.675305
2000,1.662666,1.637452
2500,1.632729,1.604057
3000,1.470732,1.588752
3500,1.428279,1.570584
4000,1.427475,1.549974
4500,1.453249,1.531518
5000,1.432528,1.514446


Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-medical/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-medical/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-medical/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-medical/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-medical/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-medical/checkpoint-2500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-medical/checkpoint-3000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-medical/checkpoint-3500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-medical/che

TrainOutput(global_step=8547, training_loss=1.473980728333298, metrics={'train_runtime': 5195.9867, 'train_samples_per_second': 52.62, 'train_steps_per_second': 1.645, 'total_flos': 3.8715396144961536e+17, 'train_loss': 1.473980728333298, 'epoch': 3.0})

In [ ]:
# Sauvegarder sur HuggingFace
model.save_pretrained("./llama3b-algerian-final")
tokenizer.save_pretrained("./llama3b-algerian-final")

from huggingface_hub import HfApi
api = HfApi()

api.create_repo(
    repo_id="abdeldjalil-smaili/llama3b-algerian-medical",
    repo_type="model",
    token=HF_WRITE_TOKEN,
    exist_ok=True
)
api.upload_folder(
    folder_path="./llama3b-algerian-final",
    repo_id="abdeldjalil-smaili/llama3b-algerian-medical",
    repo_type="model",
    token=HF_WRITE_TOKEN
)
print("Modèle sauvegardé! ✅")
print("URL: huggingface.co/abdeldjalil-smaili/llama3b-algerian-medical")

Unsloth: Restored added_tokens_decoder metadata in ./llama3b-algerian-final/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########|  195MB /  195MB            

  ...rian-final/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Modèle sauvegardé! ✅
URL: huggingface.co/djaldjoul/llama3b-algerian-medical


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

# Passer en mode inférence
FastLanguageModel.for_inference(model)

def ask_medical(question):
    prompt = f"""### Instruction:
أجب على السؤال الطبي التالي

### Input:
{question}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:" in full:
        response = full.split("### Response:")[-1].strip()
    else:
        response = full.strip()
    return response

# ── Test ──────────────────────────────────────────────
questions = [
    "وش هوما أعراض السكري؟",
    "كيفاش نعالج الصداع؟",
    "واش الحمى خطيرة؟",
    "وش هوما المأكولات لي يديرو الاسهال؟",
    "كيفاش نحمي روحي من كوفيد؟"
]

print("="*50)
print("Test LLaMA 3.2 3B — Fine-tuné")
print("="*50)

for q in questions:
    print(f"\nسؤال: {q}")
    response = ask_medical(q)
    print(f"جواب: {response}")
    print("-"*40)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test LLaMA 3.2 3B — Fine-tuné

سؤال: وش هوما أعراض السكري؟


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


جواب: العطش الشديد، خروج البول بكثرة، والبول يتحول للبني، والتعب، والصداع، ومضاعفات القلب والأوعية الدموية. | السكر التراكمي > 9% | السكر الصائم > 126 ملغ/دليل | السكر الفاطر > 5.7 ملغ/دليل | السكر التراكمي > 9% | السكر الصائم > 126 ملغ/دليل | السكر الفاطر > 5.7 ملغ/دليل | السكر التراكمي > 9% | السكر الصائم > 126 ملغ/دليل | السكر الفاطر > 5.7 ملغ/دليل | السكر التراكمي > 9% | السكر الصائم > 126 ملغ/دليل | السكر الفاطر > 5.7 ملغ/د
----------------------------------------

سؤال: كيفاش نعالج الصداع؟


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


جواب: الحصول على منافسة جيدة للعينين، تجنب الضغط العالي في العينين، استخدام الأجهزة اللاصقة للعينين مثل القراقرة أو النظارات الطبية، تناول الأطعمة التي تحتوي على فيتامين ب2. يمكن ان يقدر الطبيب علاجه بالادوية لضغوطات واحدخرا محتملة مثل الالتهابات المزمنة أو التهاب الجيوب الأنفية. يمكن ان يقدر الطبيب يوصي بعمل اختبارات راديو باش يعرف سبب الصداع ويعالجو. من الضروري ان تشاور طبيبك قبل تناول أي دواء لعلاج الصداع. ممكن يكون السبب هو التوتر والانفعالات العاطفية، ففي هاذ الحالات تقدر، تتجنب بعض القلقات مثل التوقف نشالله التد
----------------------------------------

سؤال: واش الحمى خطيرة؟


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


جواب: كاين بزاف دالأسباب تاع الحمى، وكل سبب عنده علاج لازم بيه. خاصنا نشوفو شي طبيب باش نعرفو واش هو السبب الحقيقي تاع الحمى ونعالجوها. الله يرحم. اذا كان عندكم شي سؤال آخر، روح للطبيب ديالك. الله يبارك فيك. نصحك تشوف شي طبيب باش يعاونك. الله يهديك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. الله يسهل عليك. Allah is the most merciful and compassionate of all. He forgives all sins, and His mercy is great
----------------------------------------

سؤال: وش هوما المأكولات لي يديرو الاسهال؟


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


جواب: البقوليات (الفول والبازلاء وغيرهما) واللحم المعلب والكبدة. بعض أنواع الأطعمة (الحليب و منتجات الحليب والرز والخبز). مشتقات الحامض (بيتزا والقشطة والملوف والأتاي والشاي). مشتقات الحوت. الماكلة الخفيفة (المساعرات والفلافل والمشويات). الطعام غير المصنع (الصودا والفلفل الأسود والثوم). بعض أنواع الأدوية (الأنتيكويدات). التلوث الغذائي أو وجود مواد حافظة جيدة في الأغذية. بعض أنواع الأدوية (الأنتيكويدات). التلوث الغذائي أو وجود مواد حافظة جيدة في الأغذية. بعض أنواع الأدوية (الأنتيكويدات). التلوث الغذ
----------------------------------------

سؤال: كيفاش نحمي روحي من كوفيد؟
جواب: المسح بالفم والانف باستخدام الخيط الطويل ومسح الجلد والشفاه بالماسك على الابتعاد عن العلاج من المرضى المصابين، والمشاركة في gatherings وحدد المسافة بين الناس، وتجنب لمس عينيك أو فمك بمناسبة آخر. تناول الطعام النظيف وغسل اليدين بانتظام. ممارسة الرياضة بانتظام والتغذية الصحية. اتباع الإجراءات للوقاية من الأمراض المعدية. اتباع التعليمات تاع السلامة وتنفيذ التدخلات الوقائية عند ظهور الأعراض. الحجر الصحي سافي كان الشخ

In [ ]:
def ask_medical(question):
    prompt = f"""### Instruction:
أجب على السؤال الطبي التالي

### Input:
{question}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.5,  # ← augmenté pour éviter répétitions
            no_repeat_ngram_size=4,  # ← éviter répétition de 4 mots
        )

    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:" in full:
        response = full.split("### Response:")[-1].strip()
    else:
        response = full.strip()
    return response

# ── Test ──────────────────────────────────────────────
questions = [
    "وش هوما أعراض السكري؟",
    "كيفاش نعالج الصداع؟",
    "واش الحمى خطيرة؟",
    "وش هوما المأكولات لي يديرو الاسهال؟",
    "كيفاش نحمي روحي من كوفيد؟"
]

print("="*50)
print("Test LLaMA 3.2 3B — Fix répétitions")
print("="*50)

for q in questions:
    print(f"\nسؤال: {q}")
    response = ask_medical(q)
    print(f"جواب: {response}")
    print("-"*40)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test LLaMA 3.2 3B — Fix répétitions

سؤال: وش هوما أعراض السكري؟


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


جواب: العطش الشديد، خروج البول بزاف ، وزيادة فالتبول. قد يظهر عروق زرقاء تحتو العينان وساقيك وعنقي. يمكن أن يكون عندكم جوع دائم ومضاعفات للقلب أو الكلى سافي كان المرض مزمن ويتبع نظام غذائي غير صحي لمدة طويلة. غالبًا ما كيوقع نقص فيتامينات ومعادن وحتى فقد الجسم الوزن بسبب عدم استهلاكه بشكل ممتاز للأطعمة والما والتغذية الصحية والعلميه والمثابره بالحالات الطبية اللي ممكن تسبب ارتفاع الضغط وتسار
----------------------------------------

سؤال: كيفاش نعالج الصداع؟


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


جواب: تناول الماء بزاف- تجنب الأطعمة المصنعة والدهون - ممارسة الرياضية بشكل يومي واهتمام بصحة جسمك. خدي مسكنات الالم بحسب نصائح طبيب الأسنان. استخدم فرشاة أسنان صحيّة واستعملي الفرشة بالطريقة المناسبة لتقوية عضلات الفم، وإلا سافي كان عندكم مشكل في النوم أو الحالة المستمره للصوت كالصدى ، فكر زيارة دكتور الأنف والأذان والحنجرة لتقييم وضعيتها وتتبع العلاجات لي ملائمة لها مثل حبوب مضادة للحساسية
----------------------------------------

سؤال: واش الحمى خطيرة؟


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


جواب: خاصك تشوف شي طبيب باش تعرف وين كاين النزيف. يمكن يكون فالحلق ولا فالودن، وهاد الشي يعتمد علا لوعيد ديالك. خاصنا نديرو له تيستات كاملين باش نعطيو العلاج المناسب حيت كل واحد عنده حالة مختلفة حساب شحال من نقصان فيها الدم. الله يشافيها بانتظام ويخليكم صحيحة تماما إن نشا الله. غادي ترجع لحالتيك الأصلية فأقرب وقت. ما تخافش وتجنبو الخطر اذا كان ممكن نتسناو حتى يمشي الدوا الوقائي أومبعد العملية الجر
----------------------------------------

سؤال: وش هوما المأكولات لي يديرو الاسهال؟


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


جواب: البقوليات والخضار وخاصة الإبرية. الأسماك الخافضة للحرارة، مثل سمكة الكabs. الفولاد والأطعمة المصنوعة منها ، والحليب بكميات قليلة, والمشتقات دياله, ومعجون الاسنان ، والعسل. الحبوب البرازيلية وغيرهِم. القمح الشرياني. انواع معينة تاع الدواء دون استشاره طبيبه او تحتو إشرافه طبى سريع الطراح. بعض الأدوية للحساسية بحجم حبة الجلد بالحديد أو الانيميا. البكتيريا المعوية مثل البنوكسيلا والسيفيوساكروم
----------------------------------------

سؤال: كيفاش نحمي روحي من كوفيد؟
جواب: الحفاظة بالتدابير الصحية العادية مثل غسل اليدين، استعمال الكمامة في المطلوب وعدم الإصابة بأي مرض. تابع النتائج الفحوصات المتعلقة بإجراء الاختبار للتحقق أن لا تحتاج للإلغاؤ أو تأجيل إجرائو بعدما تصاحب كورونا. تبع تعليماته الخاصة بالأطبائي المختص باش يحدد درجة الاصابات ديالك ويوجهكم إلى المستشفى سافي لزم الأمر وأخيراً اتباع التعليمات العامة للتخفية منه سرعة القذف عند الرجال علاجات طبيعية لعقاقير الدودة
----------------------------------------
